# 05c — AE ceiling versus latent propagator

One paper-focused 2D comparison for de Pablo, Reid, and mixed-temperature de Pablo. The AE ceiling decodes the true latent at each target frame; the one-step propagator predicts that latent autoregressively. Their gap separates representation error from latent-dynamics error.

In [1]:
%matplotlib inline
from pathlib import Path
import os
import sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'src'/'lss').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT/'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT/'src'))
from lss.latent.capacity import experiment_config_fingerprint, final_05_2d_propagator_config
from lss.latent.experiment import run_latent_experiment, seed_everything
from lss.utils import resolve_device
from lss.plotting import apply_editorial_style, dataset_color, measurement_style
DEVICE = resolve_device('auto')
OUTPUT = PROJECT_ROOT/'notebooks'/'results'/'05c_ae_ceiling_vs_latent_propagator'
OUTPUT.mkdir(parents=True, exist_ok=True)
apply_editorial_style()

KeyboardInterrupt: 

In [ ]:
# Shared 05 models: this is the same cache namespace used by 05b and 05d.
RESULT_ROOT = PROJECT_ROOT/'notebooks'/'results'/'05b_20net_latent_rollout'
SHARED_MODEL_ROOT = RESULT_ROOT/'capacity_sweep'
BASE_SEED = 20260710
TRAIN_COUNT, VAL_COUNT = 20, 20
LATENT_DIM = 2
LATENT_TOKENS, HIDDEN_SIZE = 32, 96
AE_TRAIN_FRAMES, DYN_TRAIN_FRAMES = 100, 100
AE_MAX_EPOCHS, DYN_MAX_EPOCHS = 80, 80
AE_PATIENCE, DYN_PATIENCE = 8, 8
ROLLOUT_STEPS = [10, 20, 50, 100, 150, 199]
FORCE_TRAIN_SHARED_MODELS = False

CASES = {
    'depablo_low_temp': {'label': 'de Pablo', 'dataset_name': 'depablo_low_temp', 'path': PROJECT_ROOT/'data'/'depablo-near-zero-temp.pt'},
    'reid': {'label': 'Reid', 'dataset_name': 'reid', 'path': PROJECT_ROOT/'data'/'reid_200_frames.pt'},
    'depablo_mixed_temp': {'label': 'de Pablo mixed-T', 'dataset_name': 'depablo_mixed_temp', 'path': PROJECT_ROOT/'data'/'depablo-10k-mix-temp.pt'},
}
for spec in CASES.values():
    assert spec['path'].exists(), f"Missing dataset: {spec['path']}"

def build_shared_case(case_key, case_spec, latent_dim):
    seed = BASE_SEED + 101 * (list(CASES).index(case_key) + 1)
    cache_dir = SHARED_MODEL_ROOT/case_key
    cache_dir.mkdir(parents=True, exist_ok=True)
    cfg = {
        'dataset_name': case_spec['dataset_name'], 'split_seed': seed, 'split_stratify_temperature': False,
        'min_train_p_ratio': None, 'device': str(DEVICE), 'pos_dim': 2, 'batch_graphs': 4,
        'frame_skip': 1, 'train_frame_start_order': 0, 'latent_dim': latent_dim,
        'latent_tokens': LATENT_TOKENS, 'hidden_size': HIDDEN_SIZE, 'autoencoder_model': 'attention',
        'edge_feature_dim': 12, 'ae_target_mode': 'normalized_delta', 'node_feature_mode': 'normalized_delta',
        'ae_max_train_frames_per_sim': AE_TRAIN_FRAMES, 'dyn_max_train_transitions_per_sim': DYN_TRAIN_FRAMES,
        'ae_max_epochs': AE_MAX_EPOCHS, 'ae_patience': AE_PATIENCE, 'ae_lr': 5e-5, 'ae_weight_decay': 1e-5,
        'dyn_max_epochs': DYN_MAX_EPOCHS, 'dyn_patience': DYN_PATIENCE, 'dyn_lr': 3e-5, 'dyn_weight_decay': 1e-4,
        'propagator_use_static_context': True, 'graph_context_dim': 16, 'propagator_context_include_temperature': False,
        'propagator_step_stride': 1, 'initial_velocity': 'zero', 'propagator_objective': 'one_step',
        'propagator_model': 'delta_mlp', 'propagator_loss': 'delta', 'propagator_standardize_latent': False,
        'propagator_rollout_eval_every_epoch': True, 'propagator_rollout_eval_horizon': 100,
        'propagator_rollout_eval_max_sims': None, 'propagator_checkpoint_metric': 'val_rollout_p_ratio_r2',
        'propagator_checkpoint_mode': 'max', 'early_stop_min_delta': 1e-5,
        'rollout_steps_grid': ROLLOUT_STEPS, 'rollout_eval_max_sims_per_split': 30,
        'temperature_pratio_window': 'full', 'temperature_pratio_estimator': 'robust',
        'temperature_pratio_min_fit_frames': 8, 'temperature_pratio_min_driven_strain_range': 1e-3,
        'temperature_pratio_smooth_window': 5, 'should_rollout': True, 'should_train_propagator': True,
        'force_train': FORCE_TRAIN_SHARED_MODELS, 'cache_path': None, 'model_seed': seed, 'repeat_idx': 1,
    }
    cache_identity = {**cfg, 'source_path': str(case_spec['path']), 'train_count': TRAIN_COUNT, 'val_count': VAL_COUNT}
    cache_tag = experiment_config_fingerprint(cache_identity)
    base_cache_path = cache_dir/f'rollout_attention_cv{latent_dim}_train{TRAIN_COUNT}_{cache_tag}.pt'
    final_propagator = final_05_2d_propagator_config(case_key) if latent_dim == 2 else None
    case_train_count, case_val_count = TRAIN_COUNT, VAL_COUNT
    if final_propagator is None:
        cfg['cache_path'] = str(base_cache_path)
    else:
        cache_name = final_propagator.pop('cache_name')
        ae_cache_name = final_propagator.pop('ae_cache_name', None)
        final_propagator.pop('seed_offset')
        case_train_count = int(final_propagator.pop('train_count', TRAIN_COUNT))
        case_val_count = int(final_propagator.pop('val_count', VAL_COUNT))
        cfg.update(final_propagator)
        cfg['pretrained_ae_cache_path'] = str(base_cache_path if ae_cache_name is None else RESULT_ROOT/'propagator_optimization'/case_key/ae_cache_name)
        cfg['cache_path'] = str(RESULT_ROOT/'propagator_optimization'/case_key/cache_name)
    source = {
        'dataset_name': case_spec['dataset_name'], 'source_name': case_spec['label'],
        'label': f"{case_spec['label']} CV{latent_dim}, train={case_train_count}", 'path': str(case_spec['path']),
        'dataset_mixture': [{'name': case_spec['dataset_name'], 'label': case_spec['label'], 'path': str(case_spec['path']), 'train_count': case_train_count, 'val_count': case_val_count}],
        'target_mode': cfg['ae_target_mode'], 'ae_target_mode': cfg['ae_target_mode'], 'node_feature_mode': cfg['node_feature_mode'],
        'latent_dim': latent_dim, 'latent_tokens': LATENT_TOKENS, 'hidden_size': HIDDEN_SIZE, 'autoencoder_model': 'attention',
        'edge_feature_dim': cfg['edge_feature_dim'], 'ae_max_train_frames_per_sim': AE_TRAIN_FRAMES,
        'dyn_max_train_transitions_per_sim': DYN_TRAIN_FRAMES, 'ae_max_epochs': AE_MAX_EPOCHS, 'ae_patience': AE_PATIENCE,
        'ae_lr': cfg['ae_lr'], 'ae_weight_decay': cfg['ae_weight_decay'], 'dyn_max_epochs': DYN_MAX_EPOCHS,
        'dyn_patience': DYN_PATIENCE, 'dyn_lr': cfg['dyn_lr'], 'dyn_weight_decay': cfg['dyn_weight_decay'],
        'model_seed': seed, 'repeat_idx': 1,
    }
    if final_propagator is not None:
        for key in ('autoencoder_model', 'hidden_size', 'latent_tokens', 'ae_max_train_frames_per_sim', 'ae_max_epochs', 'ae_patience', 'ae_lr', 'ae_weight_decay', 'dyn_lr', 'dyn_weight_decay', 'dyn_max_epochs', 'dyn_patience'):
            if key in cfg:
                source[key] = cfg[key]
    return seed, cfg, source

print('05c is plot-only: it reads the fixed three-seed complete-run statistics produced by 05b.')


In [ ]:
seeded_stats_path = RESULT_ROOT/'three_seed_ae_ceiling_vs_rollout_reid_small_ae200_v1.csv'
if not seeded_stats_path.exists():
    raise FileNotFoundError('Run the three-seed training/evaluation cells in 05b first: '+str(seeded_stats_path))
comparison = pd.read_csv(seeded_stats_path).query("split == 'test'").copy()
comparison['p_ratio_r2_display'] = comparison['p_ratio_r2'].clip(lower=0)
gap = comparison.pivot_table(index=['case', 'case_label', 'repeat_index', 'model_seed', 'rollout_steps'], columns='measurement', values='p_ratio_r2_display').reset_index()
gap['ceiling_gap'] = gap['AE ceiling'] - gap['Latent propagator']
comparison.to_csv(OUTPUT/'ae_ceiling_vs_propagator_stats.csv', index=False)
gap.to_csv(OUTPUT/'ae_ceiling_gap.csv', index=False)
display(gap.round(4))

In [ ]:
print('Three-seed 2D AE ceiling and latent propagator: mean held-out p-ratio R² ± one standard deviation.')
fig, ax = plt.subplots(figsize=(9.4, 5.8), constrained_layout=True)
for case, case_spec in CASES.items():
    frame = comparison[comparison.case.eq(case)]
    for measurement in ['AE ceiling', 'Latent propagator']:
        line_style = measurement_style(measurement)
        group = frame[frame.measurement.eq(measurement)]
        for _, seed_group in group.groupby('repeat_index'):
            seed_group = seed_group.sort_values('rollout_steps')
            ax.plot(seed_group.rollout_steps, seed_group.p_ratio_r2_display, color=dataset_color(case), linestyle=line_style['linestyle'], lw=.7, alpha=.18)
        summary = group.groupby('rollout_steps').p_ratio_r2_display.agg(['mean','std']).reset_index().sort_values('rollout_steps')
        ax.fill_between(summary.rollout_steps, (summary['mean']-summary['std']).clip(lower=0), (summary['mean']+summary['std']).clip(lower=0), color=dataset_color(case), alpha=.10, linewidth=0)
        ax.plot(summary.rollout_steps, summary['mean'], color=dataset_color(case),
                **line_style, label=f"{case_spec['label']} — {measurement}")
ax.axhline(0, color='0.2', lw=.8)
ax.set(xlabel='Target frame', ylabel='Held-out p-ratio R²', ylim=(-0.02, 1.02))
ax.legend(frameon=False, ncol=1)
fig.savefig(OUTPUT/'ae_ceiling_vs_propagator_2d_all_datasets.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT/'ae_ceiling_vs_propagator_2d_all_datasets.pdf', bbox_inches='tight')
plt.show()

## Interpretation

- Weak AE ceiling: improve the representation first.
- Strong AE ceiling with a large gap: freeze the AE and improve latent dynamics.
- Propagator close to the ceiling: dynamics are no longer the dominant limitation.